# Training: SFT, two from-scratch DPO runs, and LoRA-DPO

Accelerator: GPU T4 x2. Every cell runs a command from the repository.

## Task 1: Repository and Kaggle setup

In [ ]:
REPO_URL = 'https://github.com/your-username/cs5326-pa2'

In [ ]:
!git clone {REPO_URL} /kaggle/working/cs5326-pa2

In [ ]:
%cd /kaggle/working/cs5326-pa2

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_environment.py --require_two_gpus

In [ ]:
!PYTHONPATH=. python -m pytest -q tests

## Data preparation

In [ ]:
!python scripts/prepare_data.py --task all

## Task 3: Supervised fine-tuning

In [ ]:
!python scripts/train_sft.py --config configs/sft.yaml --resume --skip_if_complete

In [ ]:
!zip -qr /kaggle/working/sft_run_outputs.zip outputs/sft outputs/environment.json data/processed/manifest.json

## Task 4: From-scratch DPO

In [ ]:
!python scripts/train_dpo.py --config configs/dpo_hh.yaml --resume --skip_if_complete

In [ ]:
!python scripts/train_dpo.py --config configs/dpo_uf.yaml --resume --skip_if_complete

In [ ]:
!zip -qr /kaggle/working/dpo_run_outputs.zip outputs/dpo_hh outputs/dpo_uf

## Task 5: TRL DPOTrainer with LoRA

In [ ]:
!python scripts/train_lora_dpo.py --config configs/lora_dpo.yaml --resume --skip_if_complete

## Training figures and tables

In [ ]:
!python scripts/make_figures.py --output_root outputs
!python scripts/make_tables.py --output_root outputs

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

for path in sorted(Path('outputs/figures').glob('*_training_curves.png')):
    display(Image(filename=str(path)))

for path in sorted(Path('outputs/figures').glob('dpo_validation_comparison.png')):
    display(Image(filename=str(path)))

for stem in ['config_table', 'deviations_table', 'training_summary_table']:
    path = Path('outputs/tables') / f'{stem}.md'
    if path.exists():
        display(Markdown(path.read_text()))

In [ ]:
!zip -qr /kaggle/working/lora_and_report_outputs.zip outputs/lora_dpo outputs/figures outputs/tables